# Assignment #1 — Airbnb NYC 2018 EDA
This notebook reproduces the analysis and figures used in the PPT.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
csv_path = Path('Airbnb (1).csv')  # ensure the CSV is in the same folder as this notebook
df = pd.read_csv(csv_path)

# Align columns to assignment spec
col_map = {
    'id': 'Listing_id','name':'Listing_name','host_id':'Host_id','host_name':'Host_name',
    'neighbourhood_group':'Neighbourhood_group','neighbourhood':'Neighbourhood',
    'room_type':'Room_type','minimum_nights':'Minimum_nights',
    'number_of_reviews':'Total_reviews','reviews_per_month':'Reviews_per_month',
    'calculated_host_listings_count':'Host_listings_count','availability_365':'Availability_365',
    'price':'Price','latitude':'Latitude','longitude':'Longitude'
}
for src, dst in col_map.items():
    if src in df.columns and dst not in df.columns:
        df[dst] = df[src]
cols_wanted = ['Listing_id','Listing_name','Host_id','Host_name','Neighbourhood_group','Neighbourhood',
               'Latitude','Longitude','Room_type','Price','Minimum_nights','Total_reviews','Reviews_per_month',
               'Host_listings_count','Availability_365']
df = df[[c for c in cols_wanted if c in df.columns]].copy()
df.head()

In [ ]:
# Overview
df.describe(include='all')

In [ ]:
# Histogram of Price (clipped at 99th percentile for visibility)
clip = np.nanpercentile(df['Price'],99)
plt.figure(figsize=(8,5))
plt.hist(df.loc[df['Price'].between(0,clip),'Price'].dropna(), bins=40)
plt.title('Distribution of Airbnb Nightly Prices (clipped at 99th percentile)')
plt.xlabel('Price (USD)')
plt.ylabel('Listings')
plt.show()

In [ ]:
# Listings per Neighbourhood Group
ng_counts = df['Neighbourhood_group'].value_counts().sort_values(ascending=False)
ng_counts

In [ ]:
plt.figure(figsize=(8,5))
ng_counts.plot(kind='bar')
plt.title('Total Listings by Neighbourhood Group')
plt.xlabel('Neighbourhood Group')
plt.ylabel('Listings')
plt.show()

In [ ]:
# Costliest neighbourhood group (mean and median)
mean_prices = df.groupby('Neighbourhood_group')['Price'].mean().sort_values(ascending=False)
median_prices = df.groupby('Neighbourhood_group')['Price'].median().sort_values(ascending=False)
mean_prices, median_prices

In [ ]:
# Violin plot of price by neighbourhood group (clipped at 99th percentile)
order = df['Neighbourhood_group'].value_counts().index.tolist()
data = [df.loc[df['Neighbourhood_group']==ng,'Price'].dropna() for ng in order]
clip = np.nanpercentile(df['Price'],99)
data = [x[x.between(0,clip)] for x in data]
plt.figure(figsize=(9,5))
plt.violinplot(data, showmeans=True, showmedians=True)
plt.xticks(range(1,len(order)+1), order, rotation=20)
plt.title('Price Distribution by Neighbourhood Group (Violin Plot)')
plt.ylabel('Price (USD)')
plt.show()

In [ ]:
# Room type options: counts and price summary
room_counts = df['Room_type'].value_counts()
room_summary = df.groupby('Room_type')['Price'].agg(['count','mean','median','min','max']).sort_values('count', ascending=False)
room_counts, room_summary

In [ ]:
plt.figure(figsize=(7,5))
room_counts.plot(kind='bar')
plt.title('Room Type Availability (Counts)')
plt.xlabel('Room Type')
plt.ylabel('Listings')
plt.show()

In [ ]:
# Boxplot of price by room type (clipped at 99th percentile)
ordered = room_counts.index.tolist()
clip = np.nanpercentile(df['Price'],99)
data_box = [df.loc[df['Room_type']==rt,'Price'].dropna() for rt in ordered]
data_box = [x[x.between(0,clip)] for x in data_box]
plt.figure(figsize=(8,5))
plt.boxplot(data_box, labels=ordered, showmeans=True)
plt.title('Price by Room Type (Boxplot)')
plt.ylabel('Price (USD)')
plt.show()

In [ ]:
# Month-long stay feasibility: Minimum_nights <= 30 and Availability_365 >= 30
min_ok = df['Minimum_nights'] <= 30
avail_ok = df['Availability_365'] >= 30 if 'Availability_365' in df.columns else True
feasible = df[min_ok & avail_ok]
total = len(df)
feasible_count = len(feasible)
feasible_pct = 100*feasible_count/total if total else np.nan
feasible_count, total, feasible_pct

In [ ]:
plt.figure(figsize=(6,5))
values = [feasible_count, total-feasible_count]
labels = ['Feasible (>=30 nights)','Not feasible']
plt.bar(labels, values)
plt.title('Availability for 30+ Night Stays')
plt.ylabel('Listings')
plt.show()

In [ ]:
# OPTIONAL: Grouped bars — room type counts by neighbourhood group
grouped_rt = df.groupby(['Neighbourhood_group','Room_type']).size().unstack(fill_value=0)
grouped_rt

In [ ]:
import numpy as np
plt.figure(figsize=(10,6))
idx = np.arange(len(grouped_rt.index))
width = 0.8/len(grouped_rt.columns) if len(grouped_rt.columns)>0 else 0.2
for i, col in enumerate(grouped_rt.columns):
    plt.bar(idx + i*width, grouped_rt[col].values, width=width, label=str(col))
plt.xticks(idx + (len(grouped_rt.columns)-1)*width/2, grouped_rt.index, rotation=20)
plt.title('Room Type Counts by Neighbourhood Group (Grouped Bars)')
plt.xlabel('Neighbourhood Group')
plt.ylabel('Listings')
plt.legend()
plt.show()

In [ ]:
# OPTIONAL: Scatter map of Lat/Lon
plot_df = df.sample(n=min(8000, len(df)), random_state=42) if len(df)>8000 else df.copy()
plt.figure(figsize=(7,7))
if 'Neighbourhood_group' in plot_df.columns:
    for ng, sub in plot_df.groupby('Neighbourhood_group'):
        plt.scatter(sub['Longitude'], sub['Latitude'], s=5, alpha=0.6, label=str(ng))
    plt.legend(markerscale=3)
else:
    plt.scatter(plot_df['Longitude'], plot_df['Latitude'], s=5, alpha=0.6)
plt.title('NYC Airbnb Listings: Latitude vs Longitude')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()